In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import keras_ocr
from PIL import Image

In [ ]:
# Black and white the image

img = Image.open("image_50402305.JPG")
img = img.convert("1")
img.save("image_50402305_bw.JPG")

In [ ]:
pipeline = keras_ocr.pipeline.Pipeline()

images = [keras_ocr.tools.read("image_50402305_bw.JPG"), keras_ocr.tools.read("image_50402305.JPG")]

prediction_groups = pipeline.recognize(images) # Note that an image from the directory can be passed here

In [ ]:
fig, axs = plt.subplots(nrows=len(images), figsize=(10, 20))
for ax, image, predictions in zip(axs, images, prediction_groups):
    keras_ocr.tools.drawAnnotations(image=image, 
                                    predictions=predictions, 
                                    ax=ax)

# Easyocr

In [ ]:
import easyocr

reader = easyocr.Reader(['en'], gpu=False)

In [ ]:
image = keras_ocr.tools.read("https://drive.google.com/uc?id=1aX43391FACX_8bo0nxPZrSx9iGQV0oPU")
results = reader.readtext(image)

In [ ]:
PlotRes = [ (word[1], np.array(word[0])) for word in results]

In [ ]:
fig, axs = plt.subplots(nrows=1, figsize=(20, 20))
keras_ocr.tools.drawAnnotations(image=image, predictions=PlotRes, ax=axs)
plt.show()

In [ ]:
prediction_groups.append(PlotRes)
images.append(image)

HandwrittenRecog

In [ ]:
def extract(data) :
    for word in data :
        label = word[0]
        coords = word[1]
        img = img.crop(coords)
        HandwrittenRecog.

Loading weights

In [ ]:
from tensorflow.keras.layers import StringLookup
from tensorflow import keras

import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np
import os

np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
def build_model():
    # Inputs to the model
    input_img = keras.Input(shape=(image_width, image_height, 1), name="image")
    labels = keras.layers.Input(name="label", shape=(None,))

    # First conv block.
    x = keras.layers.Conv2D(
        32,
        (3, 3),
        activation="relu",
        kernel_initializer="he_normal",
        padding="same",
        name="Conv1",
    )(input_img)
    x = keras.layers.MaxPooling2D((2, 2), name="pool1")(x)

    # Second conv block.
    x = keras.layers.Conv2D(
        64,
        (3, 3),
        activation="relu",
        kernel_initializer="he_normal",
        padding="same",
        name="Conv2",
    )(x)
    x = keras.layers.MaxPooling2D((2, 2), name="pool2")(x)

    # We have used two max pool with pool size and strides 2.
    # Hence, downsampled feature maps are 4x smaller. The number of
    # filters in the last layer is 64. Reshape accordingly before
    # passing the output to the RNN part of the model.
    new_shape = ((image_width // 4), (image_height // 4) * 64)
    x = keras.layers.Reshape(target_shape=new_shape, name="reshape")(x)
    x = keras.layers.Dense(64, activation="relu", name="dense1")(x)
    x = keras.layers.Dropout(0.2)(x)

    # RNNs.
    x = keras.layers.Bidirectional(
        keras.layers.LSTM(128, return_sequences=True, dropout=0.25)
    )(x)
    x = keras.layers.Bidirectional(
        keras.layers.LSTM(64, return_sequences=True, dropout=0.25)
    )(x)

    # +2 is to account for the two special tokens introduced by the CTC loss.
    # The recommendation comes here: https://git.io/J0eXP.
    x = keras.layers.Dense(
        len(char_to_num.get_vocabulary()) + 2, activation="softmax", name="dense2"
    )(x)

    # Add CTC layer for calculating CTC loss at each step.
    output = CTCLayer(name="ctc_loss")(labels, x)

    # Define the model.
    model = keras.models.Model(
        inputs=[input_img, labels], outputs=output, name="handwriting_recognizer"
    )
    # Optimizer.
    opt = keras.optimizers.Adam()
    # Compile the model and return.
    model.compile(optimizer=opt)
    return model

In [ ]:
model = build_model()
checkpoint = tf.train.Checkpoint(model)
checkpoint.restore(os.getcwd() + "/Checkpoints/" + "HandwritingRecog").expect_partial()
